In [ ]:
from fiat_toolbox.well_being import (
    CapitalStock,
    CommunityUnit,
    IncomeConfig,
    IncomeStream,
    Liquidity,
    SimulationConfig,
    WellBeingConfig,
)

In [ ]:
housing_assets = CapitalStock(v=0.3, k=300000, pi=0.15)
rental_assets = IncomeStream(v=0.4, income=15000, recovery_time=10)

public_assets = IncomeStream(v=0.3, income=40000, recovery_time=10)
private_assets = IncomeStream(v=0.25, income=30000, recovery_time=4)

income_config = IncomeConfig(i_avg=80000, i_div=10000)

sim_config = SimulationConfig(
    eta=1.5, rho=0.06, t_max=10, dt=1 / 52, currency="$", c_min=20000, recovery_per=95.0
)

liq = Liquidity(internal=200_000)

config = WellBeingConfig(
    owner_housing=housing_assets,
    labour_assets={"Public": public_assets, "Private": private_assets},
    rental_housing=rental_assets,
    income=income_config,
    simulation=sim_config,
    liquidity=liq,
)

In [ ]:
# Create a WellBeing object and optimize the lambda value
household = CommunityUnit(config=config)
household

In [ ]:
res = household.opt_lambda()
# opt_lambda classifies its outcome via `res["status"]` (OptLambdaStatus):
# INTERIOR / FLAT / BOUNDARY_LOWER / BOUNDARY_UPPER / INFEASIBLE / FAILED.
# `res["message"]` carries a short explanation and an actionable hint
# whenever the result is non-trivial. For this config the optimum lands at
# the slowest end of the λ search range (BOUNDARY_LOWER) — the plot
# annotates the boundary with a red marker.
print(f"Status: {res['status']}")
print(f"Success: {res['success']}")
if res["message"]:
    print(f"Message: {res['message']}")
if res["success"]:
    household.plot_opt_lambda(x_type="time")

In [ ]:
household.get_losses("trapezoid")

In [ ]:
print(f"Recovery Time: {household.recovery_time:.2f} years")
print(
    f"Achieved recovery in {household.config.simulation.t_max} years: {household.achieved_recovery_percent():.2f} %"
)

In [ ]:
fig = household.plot_consumption()

## Sensitivity analysis

- **household τ** (`hh.recovery_time`) — the owner's reconstruction horizon.
- **composite τ** (`hh.composite_recovery_time()`) — the aggregate horizon across
  owner + rental + labour capital; slower components pull this up.
- **achieved recovery %** by `t_max` — how much of the asset stock has been
  rebuilt when the simulation ends.

`sensitivity_run` now accepts `c_min` as a parameter (default = 20 000) so the same
sweeps can be re-run near the tipping point to expose the non-linear dynamics.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def sensitivity_run(v=0.7, i_div=10000, savings=0, c_min=20_000):
    """Build a CommunityUnit with the given knobs, optimise, and return
    recovery-time metrics. Rental / labour structure stays fixed at the
    notebook's baseline values; only owner damage (v), diversified
    income (i_div), savings, and minimum consumption (c_min) vary.

    The `savings` knob feeds `Liquidity(internal=...)` — household-owned
    liquidity that, when depleted, contributes to the wellbeing Taylor term.

    Returns a dict with NaN values when the configuration is
    infeasible (no feasible λ in the search range).
    """
    cfg = WellBeingConfig(
        owner_housing=CapitalStock(v=v, k=300000, pi=0.15),
        rental_housing=IncomeStream(v=0.4, income=15000, recovery_time=10),
        labour_assets={
            "Public": IncomeStream(v=0.3, income=40000, recovery_time=10),
            "Private": IncomeStream(v=0.25, income=30000, recovery_time=4),
        },
        income=IncomeConfig(i_avg=80000, i_div=i_div),
        simulation=SimulationConfig(
            eta=1.5,
            rho=0.06,
            t_max=10,
            dt=1 / 52,
            currency="$",
            c_min=float(c_min),
            recovery_per=95.0,
        ),
        liquidity=Liquidity(internal=savings),
    )
    hh = CommunityUnit(cfg)
    opt = hh.opt_lambda(raise_on_fail=False)
    if not opt["success"]:
        return {
            "household_tau": np.nan,
            "composite_tau": np.nan,
            "achieved_tmax": np.nan,
            "wellbeing_loss": np.nan,
            "per_component": None,
            "status": str(opt["status"]),
        }
    hh.get_losses("trapezoid")
    return {
        "household_tau": float(hh.recovery_time),
        "composite_tau": float(hh.composite_recovery_time() or np.nan),
        "achieved_tmax": float(hh.achieved_recovery_percent() or np.nan),
        "wellbeing_loss": float(hh.total_losses["Wellbeing Loss"]),
        "per_component": hh.recovery_time_per_component,
        "status": str(opt["status"]),
    }

### Effect of damage on recovery time

Varying the damage ratio `v` on owner housing (rental and labour held fixed). As `v` grows, the reconstruction burden grows too; at some point the feasibility constraint `c(t) ≥ c_min` bites and the optimiser can no longer find a viable λ — those rows return `NaN`.

In [ ]:
v_grid = np.round(np.linspace(0.1, 0.2, 9), 2)
rows = []
for v in v_grid:
    m = sensitivity_run(v=float(v))
    rows.append(
        {
            "v": v,
            "household_τ": m["household_tau"],
            "composite_τ": m["composite_tau"],
            "achieved %": m["achieved_tmax"],
            "status": m["status"],
        }
    )
df_v = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(df_v["v"], df_v["household_τ"], marker="o", label="household τ (owner only)")
ax.plot(
    df_v["v"],
    df_v["composite_τ"],
    marker="s",
    linestyle="--",
    label="composite τ (all components)",
)
ax.set_xlabel("Owner housing damage v")
ax.set_ylabel("Recovery time (years)")
ax.set_title("Recovery horizon vs. housing damage")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
df_v

### Effect of liquidity on recovery

Savings, insurance, and external support smooth consumption through the recovery period. More liquidity → the household can tolerate higher reconstruction peaks → household τ tends to drop (faster recovery becomes feasible) and welfare loss shrinks.

In [ ]:
savings_grid = [0, 5_000, 15_000, 30_000, 60_000, 100_000, 200_000]
rows = []
for s in savings_grid:
    m = sensitivity_run(savings=float(s))
    rows.append(
        {
            "savings": s,
            "household_τ": m["household_tau"],
            "composite_τ": m["composite_tau"],
            "wellbeing_loss": m["wellbeing_loss"],
            "status": m["status"],
        }
    )
df_s = pd.DataFrame(rows)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
ax1.plot(df_s["savings"], df_s["household_τ"], marker="o", label="household τ")
ax1.plot(
    df_s["savings"],
    df_s["composite_τ"],
    marker="s",
    linestyle="--",
    label="composite τ",
)
ax1.set_xlabel("Savings ($)")
ax1.set_ylabel("Recovery time (years)")
ax1.set_title("Recovery horizon vs. liquidity")
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(df_s["savings"], df_s["wellbeing_loss"], marker="o", color="crimson")
ax2.set_xlabel("Savings ($)")
ax2.set_ylabel("Wellbeing loss ($ equivalent)")
ax2.set_title("Wellbeing loss vs. liquidity")
ax2.grid(True, alpha=0.3)
plt.tight_layout()
df_s

### Effect of diversified income

`i_div` represents non-asset-based residual income (remittances, transfers, pensions). It raises the baseline `c0 = Σ π·k + i_div`, giving the household more headroom to absorb the peak consumption drop at `t=0` — so more aggressive recovery rates become feasible.

In [ ]:
i_div_grid = np.arange(0, 20_001, 1_000)
rows = []
for i in i_div_grid:
    m = sensitivity_run(i_div=float(i))
    rows.append(
        {
            "i_div": i,
            "household_τ": m["household_tau"],
            "composite_τ": m["composite_tau"],
            "wellbeing_loss": m["wellbeing_loss"],
            "status": m["status"],
        }
    )
df_i = pd.DataFrame(rows)

infeasible = df_i[df_i["household_τ"].isna()]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(df_i["i_div"], df_i["household_τ"], marker="o", label="household τ")
ax.plot(
    df_i["i_div"], df_i["composite_τ"], marker="s", linestyle="--", label="composite τ"
)
# Mark infeasible points along the x-axis
if not infeasible.empty:
    y_bottom = ax.get_ylim()[0]
    ax.scatter(
        infeasible["i_div"],
        [y_bottom] * len(infeasible),
        marker="x",
        color="red",
        s=80,
        zorder=5,
        label="infeasible",
        clip_on=False,
    )
    for x_val in infeasible["i_div"]:
        ax.axvline(x=x_val, color="red", alpha=0.15, linestyle=":")
ax.set_xlabel("Diversified income i_div ($)")
ax.set_ylabel("Recovery time (years)")
ax.set_title("Recovery horizon vs. diversified income")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
df_i

### Joint damage × liquidity heatmap

A 2-D grid shows how the two knobs interact. The left panel is the household-only τ (owner reconstruction horizon); the right panel is the composite τ across owner + rental + labour. Blank cells are infeasible configurations where no λ in the default search range keeps `c(t) ≥ c_min`.

In [ ]:
v_axis = [0.3, 0.5, 0.7, 0.9]
s_axis = [0, 25_000, 50_000, 100_000, 200_000]

hh_mat = np.full((len(v_axis), len(s_axis)), np.nan)
co_mat = np.full((len(v_axis), len(s_axis)), np.nan)
for i, v in enumerate(v_axis):
    for j, s in enumerate(s_axis):
        m = sensitivity_run(v=float(v), savings=float(s))
        hh_mat[i, j] = m["household_tau"]
        co_mat[i, j] = m["composite_tau"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
for ax, data, title in [
    (ax1, hh_mat, "household τ (years)"),
    (ax2, co_mat, "composite τ (years)"),
]:
    sns.heatmap(
        data,
        xticklabels=[f"${s:,.0f}" for s in s_axis],
        yticklabels=[f"{v:.1f}" for v in v_axis],
        annot=True,
        fmt=".2f",
        cmap="viridis",
        ax=ax,
        cbar_kws={"label": "years"},
    )
    ax.set_xlabel("Savings")
    ax.set_title(title)
ax1.set_ylabel("Damage v")
plt.tight_layout()